# Build expected_links.json for the XUI front-end checks

Produces the ref file `check_case_links.py` consumes, with three kinds of case so the
front end is proved positively *and* negatively:

| kind | direction | what it proves |
|------|-----------|----------------|
| **positive**     | to   | the case's outgoing links ("This case is linked to") are all shown, with the right reasons |
| **negative_422** | to   | a case CCD rejected with 422 shows **no** links - the failure is real, not just an audit row |
| **incoming**     | from | a "linked-to" case shows the link in its reverse table ("Cases that link to this case") - proves the one-directional link is visible on the other side too |

"Visible" for positive/incoming means the link actually got created: both CCD refs
resolved AND the From case is terminal-SUCCESS in ack_audit.

Output is printed and written to your Results folder. Download it into
`XUI_TESTS/expected_links.json`.

In [0]:
import os, json
from datetime import datetime
from collections import OrderedDict
from pyspark.sql import functions as F
from pyspark.sql.functions import col, upper, when

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

curated_storage = f"ingest{lz_key}curated{env}"
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

ack_audit_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit"
run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print(f"env={env}  lz_key={lz_key}")

In [0]:
comb        = spark.table("hive_metastore.ariadm_active_appeals.case_link_combinations").alias("c")
ccd_reasons = spark.table("hive_metastore.ariadm_active_appeals.bronze_case_link_reasons_ccd").alias("r")
ack         = spark.read.format("delta").load(ack_audit_path)

ack_success = (
    ack.groupBy("CCDCaseReferenceNumber")
       .agg(F.max(when(upper(col("Status")) == "SUCCESS", 1).otherwise(0)).alias("ok"))
       .filter("ok = 1")
       .select(col("CCDCaseReferenceNumber").alias("success_ref"))
)

resolved = (
    comb
      .filter(col("c.CCDCaseReferenceNumberFrom").isNotNull() & col("c.CCDCaseReferenceNumberTo").isNotNull())
      .join(ack_success, col("c.CCDCaseReferenceNumberFrom") == col("success_ref"), "inner")
      .join(ccd_reasons, col("c.key") == col("r.key"), "left")
      .select(
          col("c.CCDCaseReferenceNumberFrom").alias("from_ref"),
          col("c.CaseNoFrom").alias("from_aria"),
          col("c.CCDCaseReferenceNumberTo").alias("to_ref"),
          col("c.CaseNoTo").alias("to_aria"),
          col("c.key").alias("reason_key"),
          col("r.value_en").alias("reason_label"),
          col("c.description").alias("other_description"),
      )
)
link_counts = resolved.groupBy("from_ref").count().withColumnRenamed("count", "link_count")

ack_fail = (
    ack.groupBy("CCDCaseReferenceNumber")
       .agg(F.max(when(upper(col("Status")) == "SUCCESS", 1).otherwise(0)).alias("ever_succ"),
            F.first(when(col("Error").isNotNull(), col("Error")), ignorenulls=True).alias("err"))
       .filter(col("ever_succ") == 0)
)
neg_422 = ack_fail.filter(col("err").rlike("(?i)422|unique|Collection item ID"))

from_refs = resolved.select("from_ref").distinct()
incoming = (
    resolved.select(
        col("to_ref").alias("case_ref"), col("to_aria").alias("case_aria"),
        col("from_ref").alias("linked_ref"), col("from_aria").alias("linked_aria"),
        col("reason_key"), col("reason_label"),
    ).join(from_refs, col("case_ref") == col("from_ref"), "left_anti")
)

print(f"From cases with visible links: {link_counts.count()}")
print(f"422-rejected cases:           {neg_422.count()}")
print(f"to-only (incoming) cases:     {incoming.select('case_ref').distinct().count()}")

In [0]:
N_POS, N_NEG, N_INC = 7, 3, 3

count_rows = link_counts.collect()
by_count = {}
for r in count_rows:
    by_count.setdefault(r.link_count, []).append(r.from_ref)
pos_selected = []
for tc in [1, 2, 3, 4, 5]:
    if tc in by_count:
        pos_selected.append(by_count[tc][0])
if by_count:
    pos_selected.append(by_count[max(by_count)][0])
for r in count_rows:
    if len(pos_selected) >= N_POS:
        break
    if r.from_ref not in pos_selected:
        pos_selected.append(r.from_ref)
pos_selected = pos_selected[:N_POS]

neg_rows = [(r.CCDCaseReferenceNumber, r.err) for r in neg_422.limit(N_NEG).collect()]

inc_all = incoming.collect()
inc_by_case = {}
for r in inc_all:
    inc_by_case.setdefault(r.case_ref, []).append(r)
inc_selected = list(inc_by_case.keys())[:N_INC]

print("positive:", pos_selected)
print("negative_422:", [r[0] for r in neg_rows])
print("incoming:", inc_selected)

pos_links = resolved.filter(col("from_ref").isin(pos_selected)).collect() if pos_selected else []

cases = OrderedDict()
for ref in pos_selected:
    cases[("pos", ref)] = {"ccd_ref": ref, "aria_case_no": None, "kind": "positive",
                           "direction": "to", "expected_link_count": 0, "expected_links": []}
for r in pos_links:
    c = cases[("pos", r.from_ref)]
    c["aria_case_no"] = r.from_aria
    c["expected_links"].append({"to_ccd_ref": r.to_ref, "to_aria": r.to_aria,
                                "reason_key": r.reason_key, "reason_label": r.reason_label,
                                "other_description": r.other_description})

for ref, err in neg_rows:
    cases[("neg", ref)] = {"ccd_ref": ref, "aria_case_no": None, "kind": "negative_422",
                           "direction": "to", "expected_link_count": 0, "expected_links": [],
                           "note": (err or "")[:300]}

for case_ref in inc_selected:
    rows = inc_by_case[case_ref]
    cases[("inc", case_ref)] = {
        "ccd_ref": case_ref, "aria_case_no": rows[0].case_aria, "kind": "incoming",
        "direction": "from", "expected_link_count": 0,
        "expected_links": [{"to_ccd_ref": rr.linked_ref, "to_aria": rr.linked_aria,
                            "reason_key": rr.reason_key, "reason_label": rr.reason_label,
                            "other_description": None} for rr in rows],
    }

out_cases = []
for v in cases.values():
    v["expected_link_count"] = len(v["expected_links"])
    out_cases.append(v)

output = {"env": env, "generated": datetime.utcnow().isoformat(), "cases": out_cases}
js = json.dumps(output, indent=2, default=str)
print(js)

In [0]:
out_dir = f"/Workspace/Users/{run_user}/Results/Case_Linking_Tests"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/expected_links.json"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(js)
print(f"Written to: {out_path}")
print("Download it and place it in Case_Linking_Tests/XUI_TESTS/expected_links.json")

## Notes

- **negative_422** cases expect **0** links - the script PASSes them when the case shows no
  links (confirming the 422 rejection is visible, not just logged). If a 422 case *does*
  show links, that's a real surprise and is flagged as `extra`.
- **incoming** cases are read from the reverse table (`ccd-linked-cases-from-table`,
  "Cases that link to this case"). That selector is not yet DOM-confirmed - if these rows
  come back empty, paste that table's HTML and the selector gets fixed.
- Tune `N_POS / N_NEG / N_INC` in the selection cell for more/fewer of each kind.